<a href="https://colab.research.google.com/github/Peksyaji/Alaya/blob/main/Web_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Load data
data = pd.read_csv('/content/data_berita.csv')  # Gantilah dengan path file yang kamu gunakan
data = data.dropna(subset=['Isi Berita'])

# Inisialisasi stopword remover dan stemmer
factory_stopwords = StopWordRemoverFactory()
stopwords = factory_stopwords.get_stop_words()
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()

def clean_text(text):
    # Lowercasing
    text = text.lower()
    # Menghapus angka
    text = re.sub(r'\d+', '', text)
    # Menghapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Menghapus spasi berlebih
    text = text.strip()
    # Menghapus stopwords
    text = ' '.join([word for word in text.split() if word not in stopwords])
    # Stemming
    text = stemmer.stem(text)
    return text

# Terapkan pembersihan ke kolom isi berita
data['Isi_Berita_Bersih'] = data['Isi Berita'].apply(clean_text)

# Simpan hasil
data.to_csv('dataset_bersih.csv', index=False)

In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
df = pd.read_csv('/content/dataset_bersih.csv')
dokumen = df['Isi_Berita_Bersih'].tolist()

In [3]:
from transformers import AutoTokenizer, AutoModel

# 1. Load IndoBERT Model
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [4]:
# 2. Fungsi Ekstraksi Fitur
# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Pindahkan model ke GPU
model = AutoModel.from_pretrained(model_name).to(device)

def get_bert_embeddings(texts, batch_size=32):  # Naikkan batch_size untuk GPU
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)  # Pindahkan input ke GPU

        with torch.no_grad():
            outputs = model(**inputs)

        # Average pooling dan pindahkan ke CPU (untuk kompatibilitas dengan sklearn)
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

Menggunakan device: cuda


In [5]:
# Dapatkan embedding
embeddings = get_bert_embeddings(dokumen)

In [6]:
embeddings

array([[-0.29548097,  0.5958412 ,  0.10328225, ...,  0.45709342,
        -0.18437052, -1.0344422 ],
       [-0.33810037,  0.9284333 ,  0.19256087, ...,  0.9780698 ,
         0.03307521, -0.89622486],
       [-0.2752302 ,  0.8941604 ,  0.3867591 , ...,  0.4394189 ,
        -0.24944428, -0.7246147 ],
       ...,
       [-0.6757698 ,  0.60945725,  0.7977351 , ...,  0.30322015,
        -0.5655697 , -0.8899057 ],
       [-0.34791636,  0.87173986,  0.35629055, ...,  0.20288676,
        -0.5297333 , -1.4610159 ],
       [-0.25524545,  0.7525486 ,  0.4471736 , ...,  0.16119699,
        -0.5642657 , -1.2578542 ]], dtype=float32)

In [7]:
embeddings.shape

(6942, 768)

In [8]:
# Simpan embeddings
np.save('indobert_embeddings.npy', embeddings)

In [9]:
# Load embeddings
loaded_embeddings = np.load('indobert_embeddings.npy')

In [10]:
import torch.nn as nn
import torch.optim as optim

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=128):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Siapkan data
X_tensor = torch.tensor(loaded_embeddings, dtype=torch.float32).to(device)
model_ae = AutoEncoder(input_dim=X_tensor.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model_ae.parameters(), lr=1e-3)

# Training
epochs = 50
for epoch in range(epochs):
    output = model_ae(X_tensor)
    loss = criterion(output, X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Simpan model autoencoder
torch.save(model_ae.state_dict(), 'autoencoder_model.pth')

Epoch 1/50, Loss: 0.2709
Epoch 2/50, Loss: 0.2432
Epoch 3/50, Loss: 0.1936
Epoch 4/50, Loss: 0.1501
Epoch 5/50, Loss: 0.1356
Epoch 6/50, Loss: 0.1029
Epoch 7/50, Loss: 0.0979
Epoch 8/50, Loss: 0.0984
Epoch 9/50, Loss: 0.0975
Epoch 10/50, Loss: 0.0991
Epoch 11/50, Loss: 0.0979
Epoch 12/50, Loss: 0.0938
Epoch 13/50, Loss: 0.0917
Epoch 14/50, Loss: 0.0901
Epoch 15/50, Loss: 0.0879
Epoch 16/50, Loss: 0.0869
Epoch 17/50, Loss: 0.0856
Epoch 18/50, Loss: 0.0835
Epoch 19/50, Loss: 0.0823
Epoch 20/50, Loss: 0.0813
Epoch 21/50, Loss: 0.0799
Epoch 22/50, Loss: 0.0790
Epoch 23/50, Loss: 0.0780
Epoch 24/50, Loss: 0.0766
Epoch 25/50, Loss: 0.0756
Epoch 26/50, Loss: 0.0745
Epoch 27/50, Loss: 0.0733
Epoch 28/50, Loss: 0.0724
Epoch 29/50, Loss: 0.0714
Epoch 30/50, Loss: 0.0705
Epoch 31/50, Loss: 0.0697
Epoch 32/50, Loss: 0.0689
Epoch 33/50, Loss: 0.0682
Epoch 34/50, Loss: 0.0675
Epoch 35/50, Loss: 0.0668
Epoch 36/50, Loss: 0.0661
Epoch 37/50, Loss: 0.0654
Epoch 38/50, Loss: 0.0649
Epoch 39/50, Loss: 0.

In [11]:
from sklearn.cluster import KMeans

# Gunakan representasi dari encoder
with torch.no_grad():
    encoded_repr = model_ae.encoder(X_tensor).cpu().numpy()

# Clustering
kmeans = KMeans(n_clusters=10, random_state=42)  # Jumlah topik bisa diubah
clusters = kmeans.fit_predict(encoded_repr)

# Tambahkan label ke dataframe
df['Cluster_DEC_Member'] = clusters
df.to_csv('hasil_clustering_member.csv', index=False)

In [12]:
# Bisa pakai hasil kmeans sebelumnya
centroids = kmeans.cluster_centers_

# Assign topik berdasar jarak ke centroid
from scipy.spatial.distance import cdist

distances = cdist(encoded_repr, centroids, metric='euclidean')
assigned_clusters = distances.argmin(axis=1)

df['Cluster_DEC_Centroid'] = assigned_clusters
df.to_csv('hasil_clustering_centroid.csv', index=False)

In [15]:
!pip install gensim

In [14]:
import gensim
from itertools import combination
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Persiapan untuk gensim
texts = [doc.split() for doc in df['Isi_Berita_Bersih']]
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# Ambil topik (dalam bentuk kata-kata paling dominan per cluster)
top_n_words = 10
topics = []
for i in range(kmeans.n_clusters):
    indices = df[df['Cluster_DEC_Member'] == i].index
    cluster_docs = [texts[j] for j in indices]
    all_words = sum(cluster_docs, [])
    freq = pd.Series(all_words).value_counts().head(top_n_words)
    topics.append(freq.index.tolist())

# Hitung coherence score
coherence_model = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary, coherence='c_v')
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score:.4f}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject